In [ ]:
!git clone https://github.com/exEKS/python_SaaS.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ?? 1. ???????????? ?? ?????????? ??????????????????????????????
df = pd.read_csv("/content/drive/MyDrive/csv/the_last_one.csv")
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print('Shape:', df.shape)
df.head()

In [ ]:
# ?? 2. ?????????? X ?? y ???????????????????????????????????????
TARGET = 'alarm_all_region'

LEAKAGE_COLS = [
    'alarm_count',
    'alarm_total_duration_min',
    'target_alarm_next_day',
]

# ????????? ??????? + ?????? (weather_* ???????)
X = (df
     .select_dtypes(include=[np.number, bool])
     .drop(columns=[TARGET] + LEAKAGE_COLS, errors='ignore')
     .fillna(0)
     .astype(float))  # XGBoost ???? float, ?? bool

y = df[TARGET].astype(int)

feature_cols = X.columns.tolist()
print(f' ???? ({len(feature_cols)}): {feature_cols}')
print(f'\nTarget ????????:\n{y.value_counts()}')

In [ ]:
# ?? 3. TimeSeriesSplit ?????????????????????????????????????????
rows_per_day = df.groupby('date').size().mean()
test_size = int(rows_per_day * 30)
print(f'??????/????: {rows_per_day:.0f}  ?  test_size={test_size}')

tscv = TimeSeriesSplit(
    n_splits=5,
    test_size=test_size,
    gap=0
)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators":     [100, 200, 300],
    "max_depth":        [3, 4, 6, 8],
    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5],
}

tscv_tune = TimeSeriesSplit(n_splits=3, test_size=test_size)

base_xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

search = RandomizedSearchCV(
    estimator=base_xgb,
    param_distributions=param_grid,
    n_iter=20,
    cv=tscv_tune,
    scoring="f1_weighted",
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X.values, y.values)

print("???????? ?????????:")
print(search.best_params_)
print(f"????????? F1 (cv): {search.best_score_:.3f}")


In [ ]:
# ?? 4. XGBoost ? ???????? ?? ?????? ???????????????????????????
xgb_clf = search.best_estimator_

acc_scores = []
f1_scores  = []
y_test_last = None
y_pred_last = None

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_train = X.iloc[train_idx].values
    X_test  = X.iloc[test_idx].values
    y_train = y.iloc[train_idx].values
    y_test  = y.iloc[test_idx].values

    xgb_clf.fit(X_train, y_train)
    y_pred = xgb_clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    acc_scores.append(acc)
    f1_scores.append(f1)

    if fold == 4:
        y_test_last = y_test
        y_pred_last = y_pred

    print(f"Fold {fold+1} | train={len(train_idx):,}  test={len(test_idx):,} | acc={acc:.3f}  f1={f1:.3f}")

print()
print(f"???????  | acc={np.mean(acc_scores):.3f} +/-{np.std(acc_scores):.3f}  |  f1={np.mean(f1_scores):.3f} +/-{np.std(f1_scores):.3f}")


In [ ]:
# ?? 5. Confusion matrix (fold 5 ? ?????????? ????) ????????????
print('Confusion matrix (fold 5):')
print(confusion_matrix(y_test_last, y_pred_last))
print()
print(classification_report(y_test_last, y_pred_last, zero_division=0))

In [ ]:
# ?? 6. Top-20 feature importance ??????????????????????????????
importances = pd.Series(xgb_clf.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
top20.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 features ? XGBoost', fontsize=14)
ax.set_xlabel('Feature importance')
plt.tight_layout()
plt.show()